# 4 — Linear Regression

**Purpose:** train our first learned forecasting model, then compare its errors with persistence on the **same training and validation examples**.

The task, selected space, 30-minute horizon, five-minute measurement-age rule and chronological periods are unchanged. This notebook does not open the final-test CSV or choose the final winning model.

Unlike persistence, linear regression learns numerical weights from the training examples. Its prediction is an intercept plus a weighted sum of inputs:

$$\hat y = b_0 + b_1x_1 + b_2x_2 + \cdots + b_px_p.$$

Ordinary least squares chooses those weights to minimise **training squared errors**, not validation error or MAE directly. We still report both MAE and RMSE.

**Starting choices for review:** use a repeating 24-hour representation of clock time, separate weekday flags, and raw regression outputs without rounding or clipping. These choices are explained below; they have not been tuned to make the validation scores look better. SVR should later use the same encoded information, with scaling learned only from training data.

**Files needed:** the prepared datasets, `src/evaluation.py` from Notebook 3, and the new `src/features.py`. The notebook assumes its working folder is `notebooks/`. Run in the project `.venv` kernel.

## 1 — Import the tools and shared functions

`LinearRegression` is the fitted model. `make_features` converts the five agreed inputs into numerical columns suitable for the model. The prediction and error functions from Notebook 3 remain unchanged.

Install scikit-learn in the project environment first if it is not already present. We will record the versions used in the saved results.

In [ ]:
from pathlib import Path
import sys
import json
import hashlib
import platform
from time import perf_counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from sklearn.linear_model import LinearRegression
from IPython.display import display

project_root = Path("..").resolve()
assert (project_root / "src" / "evaluation.py").is_file(), "Check the project root and file placement."
assert (project_root / "src" / "features.py").is_file(), "Copy the new src/features.py into the project."
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.evaluation import persistence_predict, regression_metrics
from src.features import make_features, INPUT_COLUMNS

print("Project root:", project_root)
print("Python:", platform.python_version(), "| scikit-learn:", sklearn.__version__)

## 2 — Load the same training and validation data

Read the files produced by Notebook 2. Do not change the dates, remove extra rows, shuffle the data, or load `test.csv`.

The checks confirm that the files still match the agreed preparation settings. If Notebook 3's saved run information is available, also check its file fingerprints. A fingerprint checks the file contents, not just its row count.

The forecast's future headcount is the **target** used for fitting or scoring. It must never be one of the prediction inputs. Observation timestamps and measurement ages are retained for auditing, not used as extra features.

In [ ]:
space_prefix = "227f1068"
data_folder = project_root / "data" / "prepared" / space_prefix
with open(data_folder / "preparation_settings.json", encoding="utf-8") as file:
    settings = json.load(file)

feature_columns = settings["feature_columns"]
target_column = settings["target_column"]
assert feature_columns == INPUT_COLUMNS
assert target_column == "target_headcount"
assert settings["space_id"].startswith(space_prefix)
assert settings["horizon_minutes"] == 30

train = pd.read_csv(data_folder / "train.csv")
validation = pd.read_csv(data_folder / "validation.csv")
datasets = {"train": train, "validation": validation}
periods = {
    "train": (settings["train_start"], settings["validation_start"]),
    "validation": (settings["validation_start"], settings["test_start"])
}
fingerprints = {}

for name, dataset in datasets.items():
    for column in ["forecast_time", "target_time", "target_headcount_observed_at"]:
        dataset[column] = pd.to_datetime(dataset[column], utc=True)
    period_start, period_end = periods[name]
    assert len(dataset) == settings["split_counts"][name]
    assert len(dataset) > 0
    assert dataset["forecast_time"].is_unique
    assert dataset["forecast_time"].is_monotonic_increasing
    assert np.isfinite(dataset[feature_columns + [target_column]].to_numpy(dtype=float)).all()
    assert (dataset["forecast_time"] >= pd.Timestamp(period_start)).all()
    assert (dataset["target_time"] < pd.Timestamp(period_end)).all()
    assert ((dataset["target_time"] - dataset["forecast_time"]) == pd.Timedelta(minutes=30)).all()
    fingerprints[name] = hashlib.sha256((data_folder / f"{name}.csv").read_bytes()).hexdigest()
    print(f"{name}: {len(dataset):,} examples checked")

baseline_record = project_root / "outputs" / space_prefix / "persistence" / "run_information.json"
if baseline_record.is_file():
    with open(baseline_record, encoding="utf-8") as file:
        baseline_information = json.load(file)
    assert fingerprints == baseline_information["csv_sha256"], "Prepared files differ from Notebook 3. Investigate before comparing."
    print("Prepared files match the saved baseline run.")

print("The final-test CSV has not been opened.")

## 3 — Represent time and weekday sensibly

The model still receives the **same five kinds of information**. Encoding changes their numerical representation, not the information available:

| Agreed input | Numerical representation |
|---|---|
| Current count and counts 15/30 minutes earlier | Three unchanged count columns |
| Time of day | Two numbers representing a position around a 24-hour clock |
| Day of week | Six yes/no columns for Tuesday through Sunday; Monday is the reference |

For clock time $h$ in decimal hours:

$$x_{\rm sin}=\sin(2\pi h/24),\qquad x_{\rm cos}=\cos(2\pi h/24).$$

Why both? Together they identify the time around a circle, so 23:59 and 00:01 are close rather than at opposite ends of a number line. This gives a linear model a smooth daily pattern. **One sine/cosine pair does not describe every possible occupancy pattern**, especially several separate daily peaks.

The weekday columns avoid assuming that Sunday (code 6) must have six times Monday's effect (code 0). Tuesday has `is_tuesday = 1` and the other flags zero. On Monday all six flags are zero. We leave out a Monday flag because the intercept already supplies the reference level.

Five conceptual inputs therefore become **11 numerical columns**. There are no new sensor measurements. This conversion is fixed: it does not learn statistics from any partition. It is a project-specific encoding choice, informed by the official scikit-learn time-feature example; it is not claimed to be the exact example used in Week 5.1.

Week 5.1 explains that a model can use transformed features and still be linear in its fitted parameters. We will not scale this initial ordinary least-squares model, keeping its coefficients in understandable units. The later SVR will need training-only scaling.

In [ ]:
X_train = make_features(train)
X_validation = make_features(validation)
y_train = train[target_column].to_numpy(dtype=float)
y_validation = validation[target_column].to_numpy(dtype=float)

assert X_train.index.equals(train.index)
assert X_validation.index.equals(validation.index)
assert list(X_train.columns) == list(X_validation.columns)
assert target_column not in X_train.columns

print("Training input table shape:", X_train.shape)
print("Validation input table shape:", X_validation.shape)
print("Encoded columns:", list(X_train.columns))
display(X_train.head())

## 4 — Fit the model using training examples only

`fit(X_train, y_train)` learns the weights from the training inputs and their known future counts. It does not use validation rows. The fitting objective is:

$$\min_{b_0,\ldots,b_p}\sum_{i\in\mathrm{training}}(y_i-\hat y_i)^2.$$

We fit once and retain this model for both partitions. `predict` later uses the learned weights; it does not update them.

The coefficient table exposes the fitted relationship. For example, a current-count coefficient of 0.4 means that, **holding all other encoded inputs fixed**, increasing the current count by one increases the forecast by 0.4 people. It does not mean a person physically causes that increase. The lagged counts are related to one another, so individual weights need cautious interpretation.

This is a first ordinary least-squares model, not a search over different algorithms or settings. Fitting time below is one measured run, not a hardware-independent performance claim.

In [ ]:
model = LinearRegression()
started = perf_counter()
model.fit(X_train, y_train)
fit_seconds = perf_counter() - started

coefficients = pd.DataFrame({
    "encoded_input": X_train.columns,
    "coefficient": model.coef_
})
print("Intercept:", model.intercept_)
print("Fitting time (seconds):", fit_seconds)
print("Training matrix rank:", model.rank_, "of", X_train.shape[1], "encoded columns")
display(coefficients)

## 5 — Predict and compare with persistence

Generate predictions for training and validation using the **same fitted model**. The shared error function is identical to Notebook 3's.

Persistence is recalculated on these same rows, so this comparison cannot accidentally use a different set of examples. It still requires no fitting. Differences between training and validation errors can reflect changed occupancy patterns as well as model limitations; a gap alone does not prove overfitting.

Outputs are evaluated as raw decimal counts. We do not round or clip negative values to improve the scores. Fractional predictions can represent an estimate, but negative occupancy is physically impossible and will be explicitly counted in the next section.

In [ ]:
encoded_by_split = {"train": X_train, "validation": X_validation}
result_rows = []
predictions_by_split = {}

for name, dataset in datasets.items():
    predicted = model.predict(encoded_by_split[name])
    observed = dataset[target_column].to_numpy(dtype=float)
    baseline_predicted = persistence_predict(dataset["headcount_now"].to_numpy())

    for model_name, forecast in [("persistence", baseline_predicted), ("linear_regression", predicted)]:
        scores = regression_metrics(observed, forecast)
        result_rows.append({
            "model": model_name,
            "split": name,
            "examples": len(dataset),
            "mae_people": scores["mae"],
            "rmse_people": scores["rmse"]
        })

    predictions = dataset[["forecast_time", "target_time", "target_headcount_observed_at"]].copy()
    predictions["observed_headcount"] = observed
    predictions["predicted_headcount"] = predicted
    predictions["residual_people"] = observed - predicted
    predictions_by_split[name] = predictions

comparison = pd.DataFrame(result_rows)
metrics = comparison[comparison["model"] == "linear_regression"].copy()
display(comparison.round(4))

validation_scores = comparison[comparison["split"] == "validation"].set_index("model")
mae_change_percent = 100 * (
    validation_scores.loc["persistence", "mae_people"]
    - validation_scores.loc["linear_regression", "mae_people"]
) / validation_scores.loc["persistence", "mae_people"]
print("Validation MAE reduction relative to persistence (%):", round(mae_change_percent, 2))
print("Positive means lower error; this is not a percentage accuracy score.")

## 6 — Check output limitations and error direction

Count physically impossible negative forecasts rather than concealing them. A later non-negative output policy would be a separate, documented choice applied consistently to the models; it is not implemented here.

Positive residuals mean we forecast too few people; negative residuals mean we forecast too many. These are diagnostic quantities, not measured ventilation consequences. Also check for individual inputs outside their observed training ranges. Those checks are only a warning: being within all separate ranges does not guarantee that an input combination was represented in training.

In [ ]:
quality_rows = []
for name, predictions in predictions_by_split.items():
    predicted = predictions["predicted_headcount"]
    residuals = predictions["residual_people"]
    quality_rows.append({
        "split": name,
        "negative_predictions": int((predicted < 0).sum()),
        "negative_percent": 100 * (predicted < 0).mean(),
        "minimum_prediction": predicted.min(),
        "mean_residual_people": residuals.mean(),
        "underprediction_percent": 100 * (residuals > 0).mean(),
        "overprediction_percent": 100 * (residuals < 0).mean()
    })
quality_summary = pd.DataFrame(quality_rows)
display(quality_summary.round(4))

range_rows = []
for column in feature_columns[:3]:
    lower = train[column].min()
    upper = train[column].max()
    outside = (validation[column] < lower) | (validation[column] > upper)
    range_rows.append({
        "input": column, "training_min": lower, "training_max": upper,
        "validation_rows_outside_range": int(outside.sum())
    })
range_summary = pd.DataFrame(range_rows)
display(range_summary)

## 7 — Measure prediction time

Time `model.predict` on the already encoded validation input table, using the same five repeats and 100 batches per repeat as Notebook 3. This is repeated prediction, not repeated training. We also time feature conversion plus prediction to expose the extra cost of preparing inputs.

CSV loading, scoring, printing and plotting are excluded. Fitting time is reported separately. Batch time divided by the number of rows is an **average throughput measure**, not a measured single-request response time. Compare all methods on the same computer and keep these timing scopes explicit.

In [ ]:
model.predict(X_validation)
encoded_batch_times = []
end_to_end_batch_times = []

for repeat in range(5):
    started = perf_counter()
    for batch in range(100):
        model.predict(X_validation)
    encoded_batch_times.append((perf_counter() - started) / 100)

    started = perf_counter()
    for batch in range(100):
        model.predict(make_features(validation))
    end_to_end_batch_times.append((perf_counter() - started) / 100)

median_seconds = float(np.median(encoded_batch_times))
timing = pd.DataFrame([{
    "model": "linear_regression",
    "split": "validation",
    "examples_per_batch": len(validation),
    "fit_seconds": fit_seconds,
    "median_batch_seconds": median_seconds,
    "microseconds_per_example": median_seconds / len(validation) * 1_000_000,
    "feature_encoding_plus_predict_batch_seconds": float(np.median(end_to_end_batch_times)),
    "repeats": 5,
    "batches_per_repeat": 100
}])
display(timing)

## 8 — Inspect the same validation week as Notebook 3

Plot the first calendar week of validation, not a week selected for favourable results. The horizontal axis is the nominal target time. Recorded targets may be up to five minutes earlier under our existing rule.

Points are not connected across unavailable periods. Empty plot regions do not mean zero occupancy. A second plot shows residuals against predictions for all eligible validation rows; patterns or large residuals may reveal limitations, but a neat-looking plot is not proof of reliability.

In [ ]:
validation_predictions = predictions_by_split["validation"]
week_start = pd.Timestamp(settings["validation_start"])
week_end = week_start + pd.Timedelta(days=7)
plot_data = validation_predictions[
    (validation_predictions["target_time"] >= week_start)
    & (validation_predictions["target_time"] < week_end)
]
plot_times = plot_data["target_time"].dt.tz_convert("Australia/Melbourne")

week_fig, ax = plt.subplots(figsize=(11, 4))
ax.scatter(plot_times, plot_data["observed_headcount"], s=16, label="Recorded target")
ax.scatter(plot_times, plot_data["predicted_headcount"], s=18, marker="x", label="Linear forecast")
ax.set_title("Linear regression: first week of validation (eligible examples only)")
ax.set_xlabel("Target time (Melbourne)")
ax.set_ylabel("Headcount (people)")
ax.legend()
week_fig.autofmt_xdate()
week_fig.tight_layout()
plt.show()

residual_fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(validation_predictions["predicted_headcount"], validation_predictions["residual_people"], s=10, alpha=0.35)
ax.axhline(0, linestyle="--", linewidth=1)
ax.set_title("Linear regression: validation residuals")
ax.set_xlabel("Predicted headcount (people)")
ax.set_ylabel("Recorded minus predicted (people)")
residual_fig.tight_layout()
plt.show()

## 9 — Save the model-development evidence

Save linear-model results in their own folder, leaving baseline outputs and all prepared data unchanged. Store the coefficients and encoding description so the fitted relationship is inspectable without loading a Python model file.

The saved comparison is provisional training/validation evidence, not the final test comparison. This notebook does not conclude that linear regression is the final chosen method. There is no search for a better encoding or output policy after inspecting these results.

In [ ]:
output_folder = project_root / "outputs" / space_prefix / "linear_regression"
output_folder.mkdir(parents=True, exist_ok=True)
metrics.to_csv(output_folder / "metrics.csv", index=False)
comparison.to_csv(output_folder / "comparison_with_persistence.csv", index=False)
coefficients.to_csv(output_folder / "coefficients.csv", index=False)
quality_summary.to_csv(output_folder / "prediction_checks.csv", index=False)
range_summary.to_csv(output_folder / "validation_input_ranges.csv", index=False)
timing.to_csv(output_folder / "timing.csv", index=False)
for name, predictions in predictions_by_split.items():
    predictions.to_csv(output_folder / f"{name}_predictions.csv", index=False)
week_fig.savefig(output_folder / "validation_first_week.png", dpi=160, bbox_inches="tight")
residual_fig.savefig(output_folder / "validation_residuals.png", dpi=160, bbox_inches="tight")

run_information = {
    "model": "linear_regression",
    "estimator": "sklearn.linear_model.LinearRegression",
    "estimator_parameters": model.get_params(),
    "training_objective": "ordinary least squares; minimise training residual sum of squares",
    "fit_partitions": ["train"],
    "evaluated_partitions": list(datasets),
    "test_data_used": False,
    "space_id": settings["space_id"],
    "horizon_minutes": settings["horizon_minutes"],
    "input_columns": feature_columns,
    "encoded_columns": list(X_train.columns),
    "encoding": "three raw counts; sine/cosine of hour/24; Tue-Sun indicators with Monday reference",
    "encoding_source_sha256": hashlib.sha256((project_root / "src" / "features.py").read_bytes()).hexdigest(),
    "scaling": "none for this ordinary least-squares model",
    "output_policy": "raw fractional predictions; no rounding, clipping or coefficient sign constraint",
    "intercept": float(model.intercept_),
    "coefficients": {column: float(value) for column, value in zip(X_train.columns, model.coef_)},
    "training_matrix_rank": int(model.rank_),
    "csv_sha256": fingerprints,
    "preparation_settings": settings,
    "python_version": platform.python_version(),
    "numpy_version": np.__version__,
    "pandas_version": pd.__version__,
    "scikit_learn_version": sklearn.__version__,
    "timing_scope": "predict on encoded validation table; encoding-plus-predict also reported; file IO excluded"
}
with open(output_folder / "run_information.json", "w", encoding="utf-8") as file:
    json.dump(run_information, file, indent=2)
print("Saved linear-regression evidence to:", output_folder)
print("The final-test CSV was not opened or evaluated.")

## Review, tests and GitHub checkpoint

Be able to explain these points in your own words:

- Persistence copies the current count; linear regression learns weights from historical training examples.
- `fit` learns those weights, while `predict` applies them without learning from validation.
- Time/day encoding changes the representation of inputs, not their availability or the study split.
- Lower MAE/RMSE on validation is useful evidence, not proof of performance in every period or an HVAC benefit.
- Negative predictions remain visible. A physical-output policy, additional sensitivity analysis and final testing remain unfinished.

Run the accumulated implementation tests from the **project-root terminal**:

```powershell
.\.venv\Scripts\python.exe -m pytest tests/test_evaluation.py tests/test_features_and_linear.py -v --junitxml=docs/linear_test_results.xml
```

These tests use small artificial data with known answers. They check the shared functions and intended fit/predict workflow, not real-world accuracy. They do not constitute complete verification of the raw data preparation.

After running, reviewing and saving, stage only the new notebook, feature source, tests, testing note and your generated test report. Do not commit `data/`, `.venv/` or bulk `outputs/`. No earlier notebook or source file needs to be overwritten.

### Sources and AI use

**Coursework:** supplied `MMA3001_Notes_Week05_1_Linear_Regression.ipynb` (least squares, transformed features, fit/predict and coefficient interpretation), Week 5.5 (metrics), and Week 5.6 (chronological validation and training-only learned preprocessing).

**Additional implementation reference:** the daily sine/cosine and weekday encoding is a project-specific proposal, not a claim that this exact occupancy encoding appeared in the course example.

- [scikit-learn LinearRegression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html)
- [scikit-learn time-related feature engineering](https://scikit-learn.org/stable/auto_examples/applications/plot_cyclical_feature_engineering.html)
- [scikit-learn common pitfalls](https://scikit-learn.org/stable/common_pitfalls.html)

This initial notebook, feature function and tests were drafted with ChatGPT. Record your review, verification, changes and decisions in the final AI reflection. Assistant-side execution is not a substitute for your local run or understanding.